In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Step 1: Imports

In [ ]:
import pandas as pd
import numpy as np
import re, os, random
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import tensorflow as tf
from transformers import RobertaTokenizer, TFRobertaForSequenceClassification
from transformers import create_optimizer

# Step 2: Set Seed

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(42)

# Step 3: Load Dataset

In [ ]:
train_df = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
test_df = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')
submission = pd.read_csv('/kaggle/input/nlp-getting-started/sample_submission.csv')

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.countplot(x=train_df['target'])
plt.title('Original Label Distribution')
plt.xticks([0, 1], ['Not Disaster', 'Disaster'])
plt.xlabel('Label')
plt.ylabel('Count')
plt.show()

# Step 4: Clean Text

In [ ]:
def clean_text(text):
    text = re.sub(r'http\S+', '', text)         # remove links
    text = re.sub(r'@\w+', '', text)            # remove mentions
    text = re.sub(r'#\w+', '', text)            # remove hashtags
    text = re.sub(r'[^A-Za-z0-9\s]+', '', text) # remove special chars
    return text.lower().strip()

train_df['text'] = train_df['text'].apply(clean_text)
test_df['text'] = test_df['text'].apply(clean_text)

# Step 5: Tokenizer

In [ ]:
MAX_LEN = 128
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

def encode_texts(texts):
    return tokenizer(
        list(texts),
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_tensors='tf'
    )

# Step 6: Prepare Inputs

In [ ]:
X_test = encode_texts(test_df['text'])

# Step 7: K-Fold Training Setup

In [ ]:
N_SPLITS = 2
BATCH_SIZE = 32
EPOCHS = 3

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
val_preds, test_preds = [], []

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df['text'], train_df['target'])):
    print(f"🚀 Fold {fold+1}")

    X_train = encode_texts(train_df.loc[train_idx, 'text'])
    X_val = encode_texts(train_df.loc[val_idx, 'text'])
    y_train = tf.convert_to_tensor(train_df.loc[train_idx, 'target'].values)
    y_val = tf.convert_to_tensor(train_df.loc[val_idx, 'target'].values)

    # ✅ Build Model
    model = TFRobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=1)
    steps_per_epoch = len(train_idx) // BATCH_SIZE
    optimizer, _ = create_optimizer(init_lr=2e-5, num_train_steps=steps_per_epoch*EPOCHS, num_warmup_steps=0)

    model.compile(optimizer=optimizer, loss=tf.keras.losses.BinaryCrossentropy(from_logits=True), metrics=['accuracy'])

    # ✅ Train
    model.fit(
        x={'input_ids': X_train['input_ids'], 'attention_mask': X_train['attention_mask']},
        y=y_train,
        validation_data=(
            {'input_ids': X_val['input_ids'], 'attention_mask': X_val['attention_mask']},
            y_val
        ),
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        verbose=1
    )

    # ✅ Inference
    val_logits = model.predict({'input_ids': X_val['input_ids'], 'attention_mask': X_val['attention_mask']}, batch_size=64).logits
    test_logits = model.predict({'input_ids': X_test['input_ids'], 'attention_mask': X_test['attention_mask']}, batch_size=64).logits

    val_preds.append(tf.sigmoid(val_logits).numpy().flatten())
    test_preds.append(tf.sigmoid(test_logits).numpy().flatten())

In [ ]:
import matplotlib.pyplot as plt

# Data from logs
fold1_acc = [0.8232, 0.8300, 0.8264]
fold1_loss = [0.4229, 0.4069, 0.4452]

fold2_acc = [0.8258, 0.8366, 0.8329]
fold2_loss = [0.4131, 0.4052, 0.4256]

epochs = [1, 2, 3]

# Plot
plt.figure(figsize=(12, 5))

# Accuracy Plot
plt.subplot(1, 2, 1)
plt.plot(epochs, fold1_acc, marker='o', label='Fold 1 Accuracy')
plt.plot(epochs, fold2_acc, marker='o', label='Fold 2 Accuracy')
plt.title('Validation Accuracy Across Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim(0.8, 0.85)
plt.grid(True)
plt.legend()

# Loss Plot
plt.subplot(1, 2, 2)
plt.plot(epochs, fold1_loss, marker='o', label='Fold 1 Loss')
plt.plot(epochs, fold2_loss, marker='o', label='Fold 2 Loss')
plt.title('Validation Loss Across Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()



# Step 8: Stack Predictions

In [ ]:
import numpy as np

# ✅ Create an empty array to store out-of-fold validation predictions
oof_preds = np.zeros(len(train_df))  # Same length as training data

# ✅ Assign each fold's val_preds to the correct indices
for i, (_, val_idx) in enumerate(skf.split(train_df['text'], train_df['target'])):
    oof_preds[val_idx] = val_preds[i]

# ✅ Safely average test predictions across all folds (test size is constant)
test_preds_mean = np.mean(test_preds, axis=0)


# Step 9: Threshold Optimization

In [ ]:
from sklearn.metrics import f1_score
import numpy as np

y_true = train_df['target'].values
best_thresh, best_f1 = 0.5, 0.0

for t in np.arange(0.3, 0.71, 0.01):
    f1 = f1_score(y_true, oof_preds > t)
    if f1 > best_f1:
        best_thresh, best_f1 = t, f1

print(f"✅ Best Threshold: {best_thresh:.2f} with F1 Score: {best_f1:.4f}")


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score
import numpy as np

# y_true and oof_preds should already be defined
thresholds = np.arange(0.1, 0.9, 0.01)
f1s = [f1_score(y_true, (oof_preds > t).astype(int)) for t in thresholds]

plt.figure(figsize=(10, 5))
plt.plot(thresholds, f1s, marker='o')
plt.title("F1 Score vs. Threshold")
plt.xlabel("Threshold")
plt.ylabel("F1 Score")
plt.grid(True)
plt.show()


In [ ]:
all_val_texts, all_val_labels, all_val_probs, all_val_preds = [], [], [], []


# Store raw text and labels for explainability later
val_texts = train_df.loc[val_idx, 'text'].values.tolist()
val_labels = train_df.loc[val_idx, 'target'].values.tolist()
val_probs = tf.sigmoid(val_logits).numpy().flatten().tolist()
val_preds_bin = (np.array(val_probs) > 0.5).astype(int).tolist()

# Accumulate
all_val_texts.extend(val_texts)
all_val_labels.extend(val_labels)
all_val_probs.extend(val_probs)
all_val_preds.extend(val_preds_bin)


In [ ]:
import random

sample_idx = random.sample(range(len(all_val_texts)), 10)
for i in sample_idx:
    print(f"Tweet: {all_val_texts[i]}")
    print(f"Actual: {all_val_labels[i]} | Predicted: {all_val_preds[i]} | Probability: {all_val_probs[i]:.3f}")
    print("-" * 60)


In [ ]:
test_texts = test_df['text'].tolist()

In [ ]:
final_probs = np.mean(test_preds, axis=0)
final_preds = (final_probs > best_thresh).astype(int)


In [ ]:
import pandas as pd

df_test_preds = pd.DataFrame({
    "text": test_texts,
    "pred": final_preds,
    "prob": final_probs
})

df_test_preds.to_csv("roberta_test_predictions.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt

plt.hist(final_probs, bins=50, color='skyblue', edgecolor='black')
plt.title("Prediction Probabilities")
plt.xlabel("Probability")
plt.ylabel("Count")
plt.grid(True)
plt.show()


In [ ]:
from transformers import RobertaTokenizer, TFRobertaForSequenceClassification

# Access attention weights
input_ids = tokenizer("There is a fire near my house!", return_tensors='tf')['input_ids']
outputs = model(input_ids, output_attentions=True)
attentions = outputs.attentions  

# Visualize attention of last layer
import seaborn as sns
sns.heatmap(attentions[-1][0][0].numpy(), cmap='viridis')  # Layer L, batch 0, head 0
plt.title("Attention Heatmap (Last Layer)")
plt.show()

# Step 10: Final Submission

In [ ]:
submission['target'] = (test_preds_mean > best_thresh).astype(int)
submission.to_csv('submission.csv', index=False)
print("🎉 Submission saved to submission.csv")